In [1]:
import medspacy


# Load sentence splitter and ConText assertion component
nlp = medspacy.load(
    medspacy_enable=[
        "medspacy_pyrush",
        "medspacy_context",
    ]
)

# Demo entity detector
ruler = nlp.add_pipe(
    "entity_ruler",
    before="medspacy_context",
    config={"phrase_matcher_attr": "LOWER"},
)

ruler.add_patterns([
    {"label": "DIAGNOSIS", "pattern": "pneumonia"},
    {"label": "DIAGNOSIS", "pattern": "breast cancer"},
    {"label": "DIAGNOSIS", "pattern": "diabetes"},
    {"label": "SYMPTOM", "pattern": "fever"},
])


def convert_assertions(entity):
    """Convert medspaCy attributes to competition labels."""
    assertions = []

    if entity._.is_negated:
        assertions.append("isNegated")

    if entity._.is_family:
        assertions.append("isFamily")

    if entity._.is_historical:
        assertions.append("isHistorical")

    return assertions


text = """
There is no evidence of pneumonia.
The patient's mother has breast cancer.
The patient has a history of diabetes.
The patient currently has fever.
"""

doc = nlp(text)

for entity in doc.ents:
    print({
        "text": entity.text,
        "type": entity.label_,
        "assertions": convert_assertions(entity),
        "medspacy": {
            "is_negated": entity._.is_negated,
            "is_family": entity._.is_family,
            "is_historical": entity._.is_historical,
            "is_uncertain": entity._.is_uncertain,
            "is_hypothetical": entity._.is_hypothetical,
        },
        "position": [entity.start_char, entity.end_char],
    })

2026-07-14 16:36:02.421 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=0] [doc 0] Token 0 'pneumonia' marked as sentence start (span begin)
2026-07-14 16:36:02.422 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=0] Token/tag mapping: [(pneumonia, True)]
2026-07-14 16:36:02.422 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=1] [doc 0] Token 0 'breast' marked as sentence start (span begin)
2026-07-14 16:36:02.423 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=1] Token/tag mapping: [(breast, True), (cancer, False)]
2026-07-14 16:36:02.423 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=2] [doc 0] Token 0 'diabetes' marked as sentence start (span begin)
2026-07-14 16:36:02.424 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=2] Token/tag mapping: [(diabetes, True)]
2026-07-14 16:36:02.425 | DE

{'text': 'pneumonia', 'type': 'DIAGNOSIS', 'assertions': ['isNegated'], 'medspacy': {'is_negated': True, 'is_family': False, 'is_historical': False, 'is_uncertain': False, 'is_hypothetical': False}, 'position': [25, 34]}
{'text': 'breast cancer', 'type': 'DIAGNOSIS', 'assertions': ['isFamily'], 'medspacy': {'is_negated': False, 'is_family': True, 'is_historical': False, 'is_uncertain': False, 'is_hypothetical': False}, 'position': [61, 74]}
{'text': 'diabetes', 'type': 'DIAGNOSIS', 'assertions': ['isHistorical'], 'medspacy': {'is_negated': False, 'is_family': False, 'is_historical': True, 'is_uncertain': False, 'is_hypothetical': False}, 'position': [105, 113]}
{'text': 'fever', 'type': 'SYMPTOM', 'assertions': [], 'medspacy': {'is_negated': False, 'is_family': False, 'is_historical': False, 'is_uncertain': False, 'is_hypothetical': False}, 'position': [141, 146]}


In [2]:
import medspacy
from medspacy.ner import TargetRule
from medspacy.context import ConTextRule

# Load the medspacy pipeline (includes tokenizer, target matcher, context, sectionizer)
nlp = medspacy.load()

# Add target concepts you want to detect (your "entities of interest")
target_matcher = nlp.get_pipe("medspacy_target_matcher")
target_rules = [
    TargetRule("pneumonia", "CONDITION"),
    TargetRule("diabetes", "CONDITION"),
]
target_matcher.add(target_rules)

# The 'medspacy_context' component is already in the pipeline by default
doc = nlp("The patient denies any history of pneumonia but has a family history of diabetes.")

for ent in doc.ents:
    print(ent.text, "-",
          "negated:", ent._.is_negated,
          "| uncertain:", ent._.is_uncertain,
          "| historical:", ent._.is_historical,
          "| hypothetical:", ent._.is_hypothetical,
          "| family:", ent._.is_family)

2026-07-14 16:36:59.957 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=5] [doc 0] Token 0 'The' marked as sentence start (span begin)
2026-07-14 16:36:59.958 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=5] Token/tag mapping: [(The, True), (patient, False), (denies, False), (any, False), (history, False), (of, False), (pneumonia, False), (but, False), (has, False), (a, False), (family, False), (history, False), (of, False), (diabetes, False), (., False)]


pneumonia - negated: True | uncertain: False | historical: True | hypothetical: False | family: False
diabetes - negated: False | uncertain: False | historical: True | hypothetical: False | family: True
